# Chaining Executions and LCEL (LangChain Expression Language)

In [1]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableSequence
from langchain_core.tracers.context import collect_runs
from langchain_openai import ChatOpenAI

In [2]:
load_dotenv()

True

In [3]:
model = "gpt-5.6-sol"

In [4]:
llm = ChatOpenAI(
    model=model,
    temperature=0.0,
    max_tokens=500,
    timeout=500,
    reasoning_effort="low",
    max_retries=2,
)

**Chaining Invocations**

In [5]:
prompt = PromptTemplate(template="Tell me something in brief about {topic}")

In [6]:
parser = StrOutputParser()

In [7]:
parser.invoke(llm.invoke(prompt.invoke({"topic": "React"})))

'React is an open-source JavaScript library for building user interfaces, especially single-page applications. Developed by Meta, it uses reusable components, JSX syntax, and a virtual DOM to efficiently update web pages. React is commonly paired with tools like Next.js, Redux, or React Router for full-featured applications.'

**Runnables**

Runnables can be 
- executed
    - invoke(), 
    - batch() 
    - and stream()
- inspected,
- and composed

In [8]:
runnables = [prompt, llm, parser]

**Execute Methods**

In [9]:
for runnable in runnables:
    print(f"{repr(runnable).split('(')[0]}")
    print(f"\tINVOKE: {runnable.invoke!r}")
    print(f"\tBATCH: {runnable.batch!r}")
    print(f"\tSTREAM: {runnable.stream!r}\n")

PromptTemplate
	INVOKE: <bound method BasePromptTemplate.invoke of PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me something in brief about {topic}')>
	BATCH: <bound method Runnable.batch of PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me something in brief about {topic}')>
	STREAM: <bound method Runnable.stream of PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me something in brief about {topic}')>

ChatOpenAI
	INVOKE: <bound method BaseChatModel.invoke of ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-5.6 Sol', 'release_date': '2026-07-09', 'last_updated': '2026-07-09', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs'

**Inspect**

In [10]:
for runnable in runnables:
    print(f"{repr(runnable).split('(')[0]}")
    print(f"\tINPUT: {runnable.get_input_schema()!r}")
    print(f"\tOUTPUT: {runnable.get_output_schema()!r}")
    print(f"\tCONFIG: {runnable.config_schema()!r}\n")

PromptTemplate
	INPUT: <class 'langchain_core.utils.pydantic.PromptInput'>
	OUTPUT: <class 'langchain_core.prompts.prompt.PromptTemplateOutput'>
	CONFIG: <class 'langchain_core.utils.pydantic.PromptTemplateConfig'>

ChatOpenAI
	INPUT: <class 'langchain_openai.chat_models.base.ChatOpenAIInput'>
	OUTPUT: <class 'langchain_openai.chat_models.base.ChatOpenAIOutput'>
	CONFIG: <class 'langchain_core.utils.pydantic.ChatOpenAIConfig'>

StrOutputParser
	INPUT: <class 'langchain_core.output_parsers.string.StrOutputParserInput'>
	OUTPUT: <class 'langchain_core.output_parsers.string.StrOutputParserOutput'>
	CONFIG: <class 'langchain_core.utils.pydantic.StrOutputParserConfig'>



**Config**

In [11]:
with collect_runs() as run_collection:
    result = llm.invoke(
        "Hello",
        config={
            "run_name": "demo_run",
            "tags": ["demo", "lcel"],
            "metadata": {"lesson": 5},
        },
    )

In [12]:
run_collection.traced_runs

[RunTree(id=01a0c52b-a2fa-7e72-8a7d-84c173bd251f, name='demo_run', run_type='llm', dotted_order='20260921T181246458544Z01a0c52b-a2fa-7e72-8a7d-84c173bd251f')]

In [13]:
run_collection.traced_runs[0].model_dump()

{'id': UUID('01a0c52b-a2fa-7e72-8a7d-84c173bd251f'),
 'name': 'demo_run',
 'start_time': datetime.datetime(2026, 9, 21, 18, 12, 46, 458544, tzinfo=datetime.timezone.utc),
 'run_type': 'llm',
 'end_time': datetime.datetime(2026, 9, 21, 18, 12, 47, 661520, tzinfo=datetime.timezone.utc),
 'extra': {'invocation_params': {'model': 'gpt-5.6-sol',
   'model_name': 'gpt-5.6-sol',
   'stream': False,
   'reasoning_effort': 'low',
   'max_completion_tokens': 500,
   '_type': 'openai-chat',
   'stop': None},
  'options': {'stop': None},
  'batch_size': 1,
  'metadata': {'lesson': 5,
   'ls_provider': 'openai',
   'ls_model_name': 'gpt-5.6-sol',
   'ls_model_type': 'chat',
   'ls_temperature': None,
   'ls_max_tokens': 500,
   'ls_integration': 'langchain_chat_model',
   'lc_versions': {'langchain-core': '1.6.3',
    'langchain': '1.4.2',
    'langchain-openai': '1.6.2'}}},
 'error': None,
 'serialized': {'lc': 1,
  'type': 'constructor',
  'id': ['langchain', 'chat_models', 'openai', 'ChatOpenAI'

**Compose Runnables**

In [14]:
chain = RunnableSequence(prompt, llm, parser)

In [15]:
type(chain)

langchain_core.runnables.base.RunnableSequence

In [16]:
chain.invoke({"topic": "React"})

'React is a JavaScript library for building interactive user interfaces, especially single-page web applications. Created by Meta, it uses reusable components, a virtual DOM for efficient updates, and a declarative approach where UI reflects application state. React is often paired with tools like Next.js, Redux, or React Router.'

In [17]:
for chunk in chain.stream({"topic": "React"}):
    print(chunk, end="", flush=True)

React is an open-source JavaScript library for building interactive user interfaces, especially single-page web applications. Developed by Meta, it uses reusable components, a virtual DOM for efficient updates, and a declarative approach to UI development. React is often paired with tools like Next.js, React Router, and state-management libraries.

In [18]:
chain.batch(
    [
        {"topic": "Python"},
        {"topic": "Data Science"},
        {"topic": "Machine Learning"},
    ]
)

['Python is a popular, beginner-friendly programming language known for its clear syntax and versatility. It’s widely used in web development, automation, data science, artificial intelligence, and scripting. Its large community and extensive library ecosystem make it suitable for both beginners and professionals.',
 'Data Science is the field of extracting useful insights from data using statistics, programming, and machine learning. It typically involves collecting and cleaning data, analyzing patterns, building predictive models, and communicating results through visualizations. Common tools include Python, R, SQL, and platforms such as Jupyter and Tableau.',
 'Machine Learning (ML) is a branch of artificial intelligence that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.\n\nCommon types include:\n- **Supervised learning:** Learns from labeled examples, such as spam detection.\n- **Unsupervised lear

In [19]:
chain.get_graph()

Graph(nodes={'b100005cc13947c99e2505a924b630c5': Node(id='b100005cc13947c99e2505a924b630c5', name='PromptInput', data=<class 'langchain_core.utils.pydantic.PromptInput'>, metadata=None), '436223b4705c4ccf8c57fbdba10ed51e': Node(id='436223b4705c4ccf8c57fbdba10ed51e', name='PromptTemplate', data=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me something in brief about {topic}'), metadata=None), '2f32b94bc8894f84b95346e7f8e12bd7': Node(id='2f32b94bc8894f84b95346e7f8e12bd7', name='ChatOpenAI', data=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-5.6 Sol', 'release_date': '2026-07-09', 'last_updated': '2026-07-09', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio

In [20]:
# chain.get_graph().print_ascii()

graph = chain.get_graph()

In [21]:
graph.print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


**Turn any function into Runnable**

In [22]:
def add_one(x: int) -> int:
    return x + 1

In [23]:
def mul_two(x: int) -> int:
    return x * 2

In [24]:
runnable_1 = RunnableLambda(add_one)
runnable_2 = RunnableLambda(mul_two)

In [25]:
sequence = runnable_1 | runnable_2
# Or equivalently:
# sequence = RunnableSequence(first=runnable_1, last=runnable_2)

In [26]:
sequence.invoke(1)

4

In [27]:
sequence.batch([1, 2, 3])

[4, 6, 8]

**Parallel Runnables**

In [28]:
parallel_chain = RunnableParallel(
    double=RunnableLambda(lambda x: x * 2),
    triple=RunnableLambda(lambda x: x * 3),
)

In [29]:
parallel_chain.invoke(3)

{'double': 6, 'triple': 9}

In [30]:
parallel_chain.get_graph().print_ascii()

+------------------------------+   
| Parallel<double,triple>Input |   
+------------------------------+   
           **        **            
         **            **          
        *                *         
  +--------+          +--------+   
  | Lambda |          | Lambda |   
  +--------+          +--------+   
           **        **            
             **    **              
               *  *                
+-------------------------------+  
| Parallel<double,triple>Output |  
+-------------------------------+  


### LCEL

In [31]:
prompt

PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me something in brief about {topic}')

In [32]:
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-5.6 Sol', 'release_date': '2026-07-09', 'last_updated': '2026-07-09', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh', 'max'], 'reasoning_effort_default': 'medium'}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001F9D651EE40>, async_client=<openai.resources.chat.completion

In [33]:
parser

StrOutputParser()

In [34]:
chain = RunnableSequence(prompt, llm, parser)
chain

PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me something in brief about {topic}')
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-5.6 Sol', 'release_date': '2026-07-09', 'last_updated': '2026-07-09', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh', 'max'], 'reasoning_effort_default': 'medium'}, client=<

In [35]:
prompt | llm | parser

PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me something in brief about {topic}')
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-5.6 Sol', 'release_date': '2026-07-09', 'last_updated': '2026-07-09', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh', 'max'], 'reasoning_effort_default': 'medium'}, client=<

In [36]:
chain = prompt | llm | parser

In [37]:
chain.invoke({"topic": "Python"})

'Python is a popular, beginner-friendly programming language known for its clear syntax and versatility. It’s widely used in web development, automation, data science, artificial intelligence, and scripting. Its large ecosystem of libraries—such as Django, pandas, and TensorFlow—helps developers build applications quickly.'